# Combining Multiple Tools in Agent Loops

A single tool call answers simple questions. Real agents **loop**: the model plans, calls one or more tools, reads observations, and decides whether to call again or answer. When several tools are available, each turn may emit **multiple parallel calls**, and later turns may **depend** on earlier observations.

This notebook builds a **multi-tool ReAct loop** for the **ShopPulse Order API** assistant with four tools — `search_docs`, `get_order`, `add`, and `list_orders` — and covers:

1. Wiring a hand-rolled agent loop with a tool registry.
2. Executing **parallel** vs **sequential** tool calls.
3. **Ordering dependencies** across turns.
4. **Iteration caps**, structured **trace logging**, and graceful fallback.

Everything runs offline with a rule-based planner. An optional cell at the end shows how to swap in a live LLM. No frameworks or prior notebooks are required.

In [ ]:
"""
ShopPulse Order API — multi-tool agent loop teaching environment.
"""

import json
import os
import re
import time
import uuid
from dataclasses import dataclass, field, asdict
from typing import Any, Callable, Literal

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


API_DOCS: list[dict[str, str]] = [
    {"id": "auth-01", "title": "Authentication", "text": "Send X-ShopPulse-Key header on every request. Rotate keys from Dashboard > Developers."},
    {"id": "orders-02", "title": "Order lifecycle", "text": "Orders: pending → paid → shipped → delivered. Cancel only within 1 hour of payment."},
    {"id": "refunds-03", "title": "Refund policy", "text": "Full refunds within 30 days of delivery. POST /v1/orders/{id}/refunds."},
    {"id": "shipping-04", "title": "Shipping updates", "text": "PATCH /v1/orders/{id}/shipping with carrier and tracking_number."},
]

ORDERS: dict[str, dict[str, Any]] = {
    "ORD-7001": {"id": "ORD-7001", "status": "paid", "total_usd": 89.99, "customer": "alice@example.com"},
    "ORD-7002": {"id": "ORD-7002", "status": "delivered", "total_usd": 24.50, "customer": "bob@example.com"},
    "ORD-7003": {"id": "ORD-7003", "status": "delivered", "total_usd": 15.00, "customer": "carol@example.com"},
}

print(f"Docs: {len(API_DOCS)} | Orders: {len(ORDERS)}")

---

## 1. The Multi-Tool Agent Loop

The core pattern is a **ReAct loop** (Reason + Act): the model reasons about what to do, acts via tools, observes results, and repeats until it can answer.

```
START
  │
  ▼
┌──────────┐
│  model   │◄────────────────┐
└────┬─────┘                 │
     │ no tool_calls         │
     ├──► END (final answer) │
     │ tool_calls (1..N)     │
     ▼                       │
┌──────────┐                 │
│  tools   │─────────────────┘
└──────────┘
     ▲
     └── iteration += 1; stop if iteration >= MAX
```

| Component | Role |
|-----------|------|
| **Model step** | Plans next action — final text or one/more `tool_calls` |
| **Tool step** | Executes every call in the current turn; returns observations |
| **Router** | If tool calls present → tools; else → end; also enforce max iterations |
| **Trace** | Log each model and tool step for debugging and evaluation |

---

## 2. Why One Tool Is Not Enough

Real support questions blend **documentation lookup**, **live data**, and **simple computation**:

| User question | Tools needed |
|---------------|--------------|
| "What is the refund policy?" | `search_docs` |
| "Status of ORD-7001?" | `get_order` |
| "Search refund docs and compute 15 + 27" | `search_docs` **and** `add` (same turn) |
| "Sum totals for ORD-7001 and ORD-7002" | `get_order` × 2 → `add` (across turns) |
| "Is ORD-7002 eligible for refund?" | `get_order` → `search_docs` (sequential) |

A multi-tool loop lets the model **compose** these capabilities instead of forcing one mega-function that tries to do everything.

---

## 3. ShopPulse Toolset — Schemas and Implementations

Four narrow tools with distinct roles:

| Tool | Role | Side effect |
|------|------|-------------|
| `search_docs` | Discover policy/how-to from documentation | Read-only |
| `get_order` | Fetch live order by ID | Read-only |
| `list_orders` | List orders filtered by status | Read-only |
| `add` | Add two numbers (totals, fees, arithmetic) | Pure compute |

In [ ]:
TOOL_SCHEMAS: list[dict[str, Any]] = [
    {
        "type": "function",
        "function": {
            "name": "search_docs",
            "description": (
                "Keyword search over ShopPulse API documentation. "
                "Use for policy and how-to questions — NOT for live order data."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search phrase, e.g. 'refund policy'."},
                    "limit": {"type": "integer", "description": "Max chunks (1-5). Default 3."},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_order",
            "description": "Fetch live order details by ID (ORD-####). Use when user names a specific order.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string", "description": "Order ID like ORD-7001."},
                },
                "required": ["order_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "list_orders",
            "description": "List orders filtered by status. Use when user asks about multiple orders or counts.",
            "parameters": {
                "type": "object",
                "properties": {
                    "status": {
                        "type": "string",
                        "enum": ["pending", "paid", "shipped", "delivered"],
                        "description": "Filter by order status.",
                    },
                },
                "required": ["status"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "add",
            "description": "Add two numbers. Use for order total sums, fee calculations, or explicit arithmetic in the question.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "First operand."},
                    "b": {"type": "number", "description": "Second operand."},
                },
                "required": ["a", "b"],
            },
        },
    },
]


def search_docs(query: str, limit: int = 3) -> str:
    terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
    scored = []
    for doc in API_DOCS:
        text = f"{doc['title']} {doc['text']}".lower()
        score = sum(1 for term in terms if term in text) if terms else 1
        scored.append((score, doc))
    scored.sort(key=lambda x: -x[0])
    hits = [d for _, d in scored[:max(1, min(limit, 5))]]
    if not hits:
        return ""
    return "\n".join(f"[{h['id']}] {h['title']}: {h['text']}" for h in hits)


def get_order(order_id: str) -> dict[str, Any]:
    order = ORDERS.get(order_id.upper())
    if not order:
        known = ", ".join(sorted(ORDERS.keys()))
        return {"error": f"Order {order_id} not found", "known_orders": known}
    return order


def list_orders(status: str) -> list[dict[str, Any]]:
    return [o for o in ORDERS.values() if o["status"] == status]


def add(a: float, b: float) -> float:
    return round(a + b, 2)


TOOL_REGISTRY: dict[str, Callable[..., Any]] = {
    "search_docs": search_docs,
    "get_order": get_order,
    "list_orders": list_orders,
    "add": add,
}

print("Registry:", list(TOOL_REGISTRY.keys()))
print("Sample search:", search_docs("refund")[:80], "...")
print("Sample order:", get_order("ORD-7001")["status"])

---

## 4. ACI Observation Envelope

In a multi-tool loop, the model reads **many** observations back-to-back. A consistent envelope helps it scan transcripts quickly:

```
[STATUS: ok | error | empty]
[TOOL: search_docs]
[SUMMARY: one-line headline]
<body>
[HINT: optional next step]
```

Every tool in the registry returns through `format_observation` so parallel results remain self-contained.

In [ ]:
def format_observation(
    tool_name: str,
    status: Literal["ok", "error", "empty"],
    body: str,
    *,
    hint: str = "",
) -> str:
    lines = [
        f"[STATUS: {status}]",
        f"[TOOL: {tool_name}]",
        f"[SUMMARY: {body.split(chr(10))[0][:80]}]",
        body,
    ]
    if hint:
        lines.append(f"[HINT: {hint}]")
    return "\n".join(lines)


def wrap_tool_result(tool_name: str, result: Any) -> str:
    """Convert raw Python return value into an ACI observation."""
    if isinstance(result, dict) and "error" in result:
        hint = ""
        if "known_orders" in result:
            hint = f"Valid order IDs: {result['known_orders']}"
        return format_observation(tool_name, "error", result["error"], hint=hint)
    if isinstance(result, str) and not result.strip():
        return format_observation(
            tool_name, "empty", "No matching documentation.",
            hint="Broaden search terms or try list_orders for live data.",
        )
    if isinstance(result, str):
        return format_observation(tool_name, "ok", result)
    if isinstance(result, (int, float)):
        return format_observation(tool_name, "ok", f"Result: {result}")
    if isinstance(result, list) and not result:
        return format_observation(tool_name, "empty", "No orders matched the filter.")
    return format_observation(tool_name, "ok", pretty(result))


print(wrap_tool_result("search_docs", search_docs("refund policy"))[:200])
print("---")
print(wrap_tool_result("get_order", get_order("ORD-9999"))[:200])

---

## 5. Agent State — Messages, Iteration, Trace

Production loops track more than chat messages:

| Field | Purpose |
|-------|--------|
| `messages` | Full conversation including `tool` role results |
| `iteration` | Model steps taken — compare against `MAX_ITERATIONS` |
| `trace` | Append-only structured events for observability |
| `hit_limit` | Whether the loop stopped due to cap rather than completion |

In [ ]:
MAX_ITERATIONS = 6
SYSTEM_PROMPT = (
    "You are a ShopPulse Order API assistant. "
    "Use search_docs for policy/how-to, get_order for specific order IDs, "
    "list_orders for status filters, and add for arithmetic. "
    "Cite doc chunk ids like [refunds-03]. Stop when you can answer."
)


@dataclass
class AgentLoopState:
    messages: list[dict[str, Any]] = field(default_factory=list)
    iteration: int = 0
    trace: list[dict[str, Any]] = field(default_factory=list)
    hit_limit: bool = False

    def append_trace(self, node: str, detail: str, **extra: Any) -> None:
        event = {"step": len(self.trace) + 1, "node": node, "detail": detail, **extra}
        self.trace.append(event)


state = AgentLoopState(
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Hello"},
    ]
)
state.append_trace("init", "state created")
print(f"Messages: {len(state.messages)} | Trace events: {len(state.trace)}")

---

## 6. Executing Tool Calls — The Tool Step

Each model turn may include **zero or more** tool calls. The tool step must:

1. Loop over **every** `tool_call` in the assistant message (never assume just one).
2. Parse JSON arguments, validate required fields, dispatch via registry.
3. Wrap results in ACI observations.
4. Append a `role: tool` message per call with matching `tool_call_id`.

In [ ]:
REQUIRED_FIELDS: dict[str, list[str]] = {
    "search_docs": ["query"],
    "get_order": ["order_id"],
    "list_orders": ["status"],
    "add": ["a", "b"],
}


def validate_args(name: str, args: dict[str, Any]) -> tuple[bool, str]:
    missing = [f for f in REQUIRED_FIELDS.get(name, []) if f not in args]
    if missing:
        return False, f"missing required fields: {missing}"
    return True, "ok"


def execute_single_tool_call(tool_call: dict[str, Any]) -> dict[str, Any]:
    """Execute one tool call; return OpenAI-style tool message."""
    fn = tool_call["function"]
    name = fn["name"]
    call_id = tool_call["id"]

    if name not in TOOL_REGISTRY:
        content = format_observation(
            name, "error",
            f"Unknown tool: {name}. Available: {', '.join(TOOL_REGISTRY)}.",
        )
        return {"role": "tool", "tool_call_id": call_id, "name": name, "content": content}

    try:
        args = json.loads(fn["arguments"]) if isinstance(fn["arguments"], str) else fn["arguments"]
    except json.JSONDecodeError as exc:
        content = format_observation(name, "error", f"Invalid JSON arguments: {exc}")
        return {"role": "tool", "tool_call_id": call_id, "name": name, "content": content}

    ok, msg = validate_args(name, args)
    if not ok:
        content = format_observation(name, "error", msg, hint="Fix arguments and retry.")
        return {"role": "tool", "tool_call_id": call_id, "name": name, "content": content}

    try:
        result = TOOL_REGISTRY[name](**args)
        content = wrap_tool_result(name, result)
    except Exception as exc:
        content = format_observation(name, "error", f"Execution failed: {exc}")

    return {"role": "tool", "tool_call_id": call_id, "name": name, "content": content}


def execute_tool_batch(
    assistant_msg: dict[str, Any],
    state: AgentLoopState,
) -> list[dict[str, Any]]:
    """Execute all tool_calls in one assistant turn; log trace events."""
    results = []
    for tc in assistant_msg.get("tool_calls", []):
        t0 = time.perf_counter()
        tool_msg = execute_single_tool_call(tc)
        ms = round((time.perf_counter() - t0) * 1000, 2)
        name = tc["function"]["name"]
        status_line = tool_msg["content"].split("\n")[0]
        state.append_trace(
            "tools", f"{name} executed",
            tool=name, status=status_line, ms=ms,
            args=tc["function"].get("arguments", ""),
        )
        results.append(tool_msg)
    return results


# Dry-run: parallel search + add without LLM
demo_assistant = {
    "role": "assistant",
    "content": None,
    "tool_calls": [
        {
            "id": "call_1", "type": "function",
            "function": {"name": "search_docs", "arguments": json.dumps({"query": "refund"})},
        },
        {
            "id": "call_2", "type": "function",
            "function": {"name": "add", "arguments": json.dumps({"a": 15, "b": 27})},
        },
    ],
}
demo_state = AgentLoopState()
demo_msgs = execute_tool_batch(demo_assistant, demo_state)
print(f"Parallel results: {len(demo_msgs)}")
for m in demo_msgs:
    print(f"  {m['name']}: {m['content'].split(chr(10))[0]}")
print("Trace:", pretty(demo_state.trace))

---

## 7. Parallel Tool Calls

Modern models often emit **multiple** `tool_calls` in a single turn. Execute them **all** before the next model call.

| Pattern | Example | Safe when |
|---------|---------|-----------|
| **Parallel** | `search_docs` + `add` in same turn | Tools are independent |
| **Sequential (next turn)** | `get_order` → `add` using totals | Second call needs first result |
| **Mixed** | Model batches reads; writes alone | Writes isolated per turn |

**Rule:** Never assume a single tool call — always loop `assistant_msg.tool_calls`.

In this notebook, `execute_tool_batch` runs calls sequentially in Python, but they are logically **parallel** from the model's perspective: all observations arrive before the next planning step.

In [ ]:
def make_tool_call(name: str, args: dict[str, Any], call_id: str | None = None) -> dict[str, Any]:
    return {
        "id": call_id or f"call_{uuid.uuid4().hex[:8]}",
        "type": "function",
        "function": {"name": name, "arguments": json.dumps(args)},
    }


def demo_parallel_independent() -> None:
    """search_docs and get_order are independent — safe to batch."""
    state = AgentLoopState()
    assistant = {
        "role": "assistant", "content": None,
        "tool_calls": [
            make_tool_call("search_docs", {"query": "API key"}),
            make_tool_call("get_order", {"order_id": "ORD-7001"}),
        ],
    }
    msgs = execute_tool_batch(assistant, state)
    print("Parallel independent batch:")
    for m in msgs:
        print(f"  {m['name']}: {m['content'].split(chr(10))[2][:60]}...")


demo_parallel_independent()

---

## 8. Ordering Dependencies Across Turns

Some tasks **require** sequential tool use across multiple loop iterations:

```
Task: "Sum totals for ORD-7001 and ORD-7002"

Turn 1: get_order(ORD-7001) + get_order(ORD-7002)   — parallel reads
Turn 2: add(89.99, 24.50)                           — needs turn 1 totals
Turn 3: final answer                                — no tool calls
```

```
Task: "Is ORD-7002 eligible for a refund?"

Turn 1: get_order(ORD-7002)        — status=delivered
Turn 2: search_docs("refund")      — policy says 30 days after delivery
Turn 3: final answer with citation [refunds-03]
```

The agent loop handles this naturally — the model sees Turn 1 observations before planning Turn 2.

In [ ]:
def simulate_dependency_chain() -> tuple[list[str], list[dict[str, Any]]]:
    """Two-turn dependency without LLM: search → get_order inferred from hit."""
    state = AgentLoopState()
    trace_lines: list[str] = []

    # Turn 1 — search for auth docs
    ai1 = {"role": "assistant", "content": None, "tool_calls": [make_tool_call("search_docs", {"query": "API key"})]}
    msgs1 = execute_tool_batch(ai1, state)
    trace_lines.append(f"Turn 1: {msgs1[0]['name']} → {msgs1[0]['content'].split(chr(10))[0]}")

    # Turn 2 — model infers user also wants a specific order (simulated)
    ai2 = {"role": "assistant", "content": None, "tool_calls": [make_tool_call("get_order", {"order_id": "ORD-7001"})]}
    msgs2 = execute_tool_batch(ai2, state)
    trace_lines.append(f"Turn 2: {msgs2[0]['name']} → status from observation")

    return trace_lines, state.trace


lines, tr = simulate_dependency_chain()
for line in lines:
    print(line)
print(f"Structured trace events: {len(tr)}")

---

## 9. MultiToolPlanner — Offline Multi-Step Simulator

`MultiToolPlanner` mimics what a live LLM returns across **multiple turns**, including parallel batches and follow-up calls based on prior observations.

In [ ]:
class MultiToolPlanner:
    """Rule-based planner that supports multi-turn and parallel tool calls."""

    def plan(self, user_text: str, messages: list[dict[str, Any]]) -> dict[str, Any]:
        tool_msgs = [m for m in messages if m.get("role") == "tool"]
        assistant_tool_turns = sum(
            1 for m in messages if m.get("role") == "assistant" and m.get("tool_calls")
        )

        # --- After tools ran: decide next action or finalize ---
        if tool_msgs:
            return self._continue_or_finalize(user_text, tool_msgs, assistant_tool_turns)

        # --- Turn 1 planning from user text ---
        return self._initial_plan(user_text)

    def _initial_plan(self, user_text: str) -> dict[str, Any]:
        t = user_text.lower()
        numbers = [float(x) for x in re.findall(r"\d+(?:\.\d+)?", user_text)]
        order_ids = [m.group(0).upper() for m in re.finditer(r"ord-\d+", t, re.I)]

        # Parallel: search + arithmetic in same question
        if "search" in t and "compute" in t and len(numbers) >= 2:
            query = re.sub(r"and compute.*", "", user_text, flags=re.I).replace("search for", "").strip()
            return self._assistant_tools([
                make_tool_call("search_docs", {"query": query or "refund policy"}),
                make_tool_call("add", {"a": numbers[0], "b": numbers[1]}),
            ])

        # Parallel: two order lookups for sum question
        if len(order_ids) >= 2 and any(w in t for w in ("sum", "total", "combine", "add")):
            return self._assistant_tools([
                make_tool_call("get_order", {"order_id": order_ids[0]}),
                make_tool_call("get_order", {"order_id": order_ids[1]}),
            ])

        # Single order + policy question → get_order first
        if order_ids and any(w in t for w in ("refund", "eligible", "policy")):
            return self._assistant_tools([make_tool_call("get_order", {"order_id": order_ids[0]})])

        if order_ids:
            return self._assistant_tools([make_tool_call("get_order", {"order_id": order_ids[0]})])

        if any(w in t for w in ("delivered", "paid", "shipped", "pending")) and "how many" in t:
            status = next(s for s in ("delivered", "paid", "shipped", "pending") if s in t)
            return self._assistant_tools([make_tool_call("list_orders", {"status": status})])

        if any(w in t for w in ("policy", "api key", "auth", "how do", "refund rules", "shipping")):
            return self._assistant_tools([make_tool_call("search_docs", {"query": user_text, "limit": 2})])

        if len(numbers) >= 2 and any(w in t for w in ("add", "sum", "compute", "plus")):
            return self._assistant_tools([make_tool_call("add", {"a": numbers[0], "b": numbers[1]})])

        return {"role": "assistant", "content": "Ask about ShopPulse orders (ORD-####), API docs, or arithmetic."}

    def _continue_or_finalize(
        self, user_text: str, tool_msgs: list[dict[str, Any]], turns: int,
    ) -> dict[str, Any]:
        t = user_text.lower()
        last_names = [m.get("name", "") for m in tool_msgs]

        # After parallel get_order × 2 → add totals
        if turns == 1 and last_names.count("get_order") >= 2:
            totals = []
            for m in tool_msgs:
                if m.get("name") == "get_order" and "total_usd" in m["content"]:
                    match = re.search(r'"total_usd":\s*([\d.]+)', m["content"])
                    if match:
                        totals.append(float(match.group(1)))
            if len(totals) == 2:
                return self._assistant_tools([make_tool_call("add", {"a": totals[0], "b": totals[1]})])

        # After get_order for refund eligibility → search policy
        if turns == 1 and "get_order" in last_names and any(w in t for w in ("refund", "eligible")):
            return self._assistant_tools([make_tool_call("search_docs", {"query": "refund policy delivered"})])

        # Finalize
        return {"role": "assistant", "content": self._synthesize_answer(user_text, tool_msgs)}

    def _assistant_tools(self, tool_calls: list[dict[str, Any]]) -> dict[str, Any]:
        return {"role": "assistant", "content": None, "tool_calls": tool_calls}

    def _synthesize_answer(self, user_text: str, tool_msgs: list[dict[str, Any]]) -> str:
        parts = []
        for m in tool_msgs:
            name = m.get("name", "tool")
            body = m["content"].split("\n", 3)[-1][:300]
            parts.append(f"From {name}:\n{body}")
        return "\n\n".join(parts) if parts else "Done."


planner = MultiToolPlanner()
print("Planner ready — supports parallel and multi-turn chains.")

---

## 10. MultiToolAgent — Full ReAct Loop

`MultiToolAgent` wires planner → tool batch → repeat until the model returns text or `MAX_ITERATIONS` is hit.

In [ ]:
@dataclass
class MultiToolAgent:
    planner: MultiToolPlanner = field(default_factory=MultiToolPlanner)
    max_iterations: int = MAX_ITERATIONS
    state: AgentLoopState = field(default_factory=AgentLoopState)

    def run(self, user_message: str) -> dict[str, Any]:
        self.state = AgentLoopState(
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_message},
            ]
        )

        while self.state.iteration < self.max_iterations:
            self.state.iteration += 1
            decision = self.planner.plan(user_message, self.state.messages)
            self.state.messages.append(decision)

            tool_calls = decision.get("tool_calls", [])
            self.state.append_trace(
                "model",
                f"iter={self.state.iteration}",
                tool_calls=[tc["function"]["name"] for tc in tool_calls],
            )

            if decision.get("content") and not tool_calls:
                return self._result(decision["content"])

            tool_msgs = execute_tool_batch(decision, self.state)
            self.state.messages.extend(tool_msgs)

        self.state.hit_limit = True
        return self._result(
            "Iteration limit reached. Partial results available in trace.",
            hit_limit=True,
        )

    def _result(self, answer: str, hit_limit: bool = False) -> dict[str, Any]:
        return {
            "answer": answer,
            "iterations": self.state.iteration,
            "trace": self.state.trace,
            "hit_limit": hit_limit or self.state.hit_limit,
            "message_count": len(self.state.messages),
        }


def run_demo(label: str, query: str) -> None:
    print("=" * 72)
    print(f"{label}")
    print(f"Q: {query}")
    agent = MultiToolAgent()
    result = agent.run(query)
    print(f"Iterations: {result['iterations']} | Hit limit: {result['hit_limit']}")
    print(f"Tool chain: {[e.get('tool_calls') or e.get('tool') for e in result['trace'] if e['node'] in ('model', 'tools')]}")
    print(f"A: {result['answer'][:350]}..." if len(result['answer']) > 350 else f"A: {result['answer']}")
    print()

In [ ]:
run_demo("Single tool — doc search", "What is the refund policy for delivered orders?")
run_demo("Parallel — search + arithmetic", "Search for refund policy docs and compute 15 + 27")
run_demo("Multi-turn — sum order totals", "Sum the totals for ORD-7001 and ORD-7002")
run_demo("Sequential — refund eligibility", "Is ORD-7002 eligible for a refund?")
run_demo("List filter", "How many delivered orders are there?")

---

## 11. Structured Trace Logging

Production traces should be **machine-readable** for dashboards and eval pipelines:

```json
{"step": 1, "node": "model", "detail": "iter=1", "tool_calls": ["search_docs", "add"]}
{"step": 2, "node": "tools", "tool": "search_docs", "status": "[STATUS: ok]", "ms": 1.2}
{"step": 3, "node": "tools", "tool": "add", "status": "[STATUS: ok]", "ms": 0.1}
{"step": 4, "node": "model", "detail": "iter=2", "tool_calls": []}
```

| Field | Why |
|-------|-----|
| `step` | Order in loop |
| `tool_calls` | What the model chose |
| `status` | ok / error / empty from observation |
| `ms` | Latency per tool |
| `iteration` | Compare against MAX |

In [ ]:
@dataclass
class TraceEvent:
    step: int
    node: str
    detail: str
    ms: float = 0.0
    extra: dict[str, Any] = field(default_factory=dict)


class TraceLogger:
    def __init__(self) -> None:
        self.events: list[TraceEvent] = []
        self._step = 0

    def log(self, node: str, detail: str, ms: float = 0.0, **extra: Any) -> None:
        self._step += 1
        self.events.append(TraceEvent(self._step, node, detail, ms, extra))

    def to_json_lines(self) -> str:
        return "\n".join(
            json.dumps({**asdict(e), **e.extra}) for e in self.events
        )

    def tool_summary(self) -> dict[str, int]:
        counts: dict[str, int] = {}
        for e in self.events:
            if e.node == "tools" and "tool" in e.extra:
                counts[e.extra["tool"]] = counts.get(e.extra["tool"], 0) + 1
        return counts


logger = TraceLogger()
t0 = time.perf_counter()
_ = search_docs("pytest")
logger.log("tools", "search_docs ok", (time.perf_counter() - t0) * 1000, tool="search_docs", status="ok")
logger.log("model", "iter=2 tool_calls=[]")
print(logger.to_json_lines())
print("Tool counts:", logger.tool_summary())

---

## 12. Max Iterations and Graceful Fallback

Without a cap, a confused model can loop forever — burning tokens and latency.

| Scenario | MAX_ITERATIONS guidance |
|----------|-------------------------|
| Doc Q&A | 4–6 |
| Multi-tool research | 6–10 |
| Open-ended coding | 12–15 with human gate |
| Production default | Hard cap + cost guard |

Hitting the cap should return a **partial answer** plus the trace — not silence.

In [ ]:
class RunawayPlanner:
    """Always requests another tool — for testing iteration cap."""

    def plan(self, user_text: str, messages: list[dict[str, Any]]) -> dict[str, Any]:
        if any(m.get("role") == "tool" for m in messages):
            return {"role": "assistant", "content": "Should not reach here if cap works."}
        return {
            "role": "assistant", "content": None,
            "tool_calls": [make_tool_call("search_docs", {"query": "refund"})],
        }


class StubbornPlanner(RunawayPlanner):
    """Keeps calling tools every turn — triggers hit_limit."""

    def plan(self, user_text: str, messages: list[dict[str, Any]]) -> dict[str, Any]:
        return {
            "role": "assistant", "content": None,
            "tool_calls": [make_tool_call("search_docs", {"query": "loop"})],
        }


def invoke_with_fallback(agent: MultiToolAgent, question: str) -> dict[str, Any]:
    result = agent.run(question)
    if result["hit_limit"]:
        result["answer"] = (
            "Iteration limit reached. "
            f"Completed {result['iterations']} model steps. "
            "See trace for partial tool results."
        )
    return result


capped = MultiToolAgent(planner=StubbornPlanner(), max_iterations=3)
fb = invoke_with_fallback(capped, "trigger runaway")
print(f"Hit limit: {fb['hit_limit']} | Iterations: {fb['iterations']}")
print(f"Trace steps: {len(fb['trace'])}")
print(f"Answer: {fb['answer']}")

---

## 13. Tool Selection Patterns in Multi-Tool Loops

| User intent | Expected tool chain |
|-------------|---------------------|
| "What is the refund policy?" | `search_docs` → answer with [refunds-03] |
| "Status of ORD-7001" | `get_order` → answer |
| "Search docs and compute 15+27" | `search_docs` + `add` (parallel) → answer |
| "Sum ORD-7001 and ORD-7002 totals" | `get_order` × 2 → `add` → answer |
| "Is ORD-7002 refund eligible?" | `get_order` → `search_docs` → answer |
| Off-topic | No tools — direct answer |

Eval sets should cover **each chain** — not only single-tool turns.

---

## 14. Parallel vs Sequential — Decision Table

| Condition | Execute |
|-----------|---------|
| All reads, no shared args | Parallel OK in one turn |
| Write + read same resource | Sequential |
| Second call uses inferred value from first | Sequential (next turn) |
| Provider limit on parallel calls | Batch or sequential fallback |
| One bad arg in parallel batch | Other results still return — model must handle partial success |

Your executor should return **one observation per tool_call_id** even when sibling calls fail.

---

## 15. Streaming Agent Steps (Preview)

For UX, stream **intermediate steps** to the client while the loop runs:

| Event | Payload |
|-------|---------|
| `tool_start` | `{name, args}` |
| `tool_end` | `{name, ms, preview}` |
| `token` | partial answer text |
| `done` | final answer + trace |

The loop structure is identical — only the delivery mechanism changes from batch to async iterator.

In [ ]:
def simulate_stream_events(query: str) -> list[dict[str, Any]]:
    """Replay stream events from an agent run for UI prototyping."""
    agent = MultiToolAgent()
    result = agent.run(query)
    events: list[dict[str, Any]] = []
    for e in result["trace"]:
        if e["node"] == "model":
            events.append({"event": "model", "iter": e["detail"], "tool_calls": e.get("tool_calls", [])})
        elif e["node"] == "tools":
            events.append({"event": "tool_end", "name": e.get("tool"), "ms": e.get("ms", 0)})
    events.append({"event": "done", "answer_preview": result["answer"][:80]})
    return events


for event in simulate_stream_events("Search for refund policy docs and compute 15 + 27"):
    print(event)

---

## 16. Debugging Failed Multi-Tool Runs

| Symptom | Check |
|---------|-------|
| Infinite loop | `MAX_ITERATIONS`, model always calls tools |
| Wrong tool first | System prompt + tool descriptions |
| Parallel call error | One bad arg fails — inspect all tool messages |
| Missing citation | Trace shows search ran? Observation empty? |
| Stuck after turn 1 | Model not reading observations — check ACI format |

Always inspect the **full message list**, not just the final answer.

In [ ]:
def summarize_messages(messages: list[dict[str, Any]]) -> list[str]:
    lines = []
    for m in messages:
        role = m.get("role", "?")
        if m.get("tool_calls"):
            names = [tc["function"]["name"] for tc in m["tool_calls"]]
            lines.append(f"{role}: tool_calls={names}")
        elif role == "tool":
            preview = str(m.get("content", ""))[:60].replace("\n", " ")
            lines.append(f"tool[{m.get('name')}]: {preview}...")
        else:
            preview = str(m.get("content", ""))[:60]
            lines.append(f"{role}: {preview}")
    return lines


agent = MultiToolAgent()
result = agent.run("Sum the totals for ORD-7001 and ORD-7002")
print("Message transcript:")
for line in summarize_messages(agent.state.messages):
    print(f"  {line}")

---

## 17. Eval Metrics for Multi-Tool Loops

| Metric | How to measure |
|--------|----------------|
| **Chain accuracy** | Expected tool sequence matches trace |
| **Parallel batch correctness** | All independent calls in one turn when possible |
| **Turn efficiency** | Fewer iterations is cheaper (if task succeeds) |
| **Recovery rate** | Model fixes args after parseable error |
| **Cap hit rate** | How often `MAX_ITERATIONS` triggers — should be near zero |

In [ ]:
EVAL_CASES = [
    ("What is the refund policy?", ["search_docs"]),
    ("Status of ORD-7001", ["get_order"]),
    ("Search for refund docs and compute 15 + 27", ["search_docs", "add"]),
    ("Sum totals for ORD-7001 and ORD-7002", ["get_order", "get_order", "add"]),
]

print(f"{'Prompt':<45} {'Expected chain':<30} {'Actual tools'}")
print("-" * 95)
for prompt, expected in EVAL_CASES:
    agent = MultiToolAgent()
    result = agent.run(prompt)
    actual = [e.get("tool") for e in result["trace"] if e["node"] == "tools"]
    match = "✓" if actual == expected else "✗"
    print(f"{prompt:<45} {str(expected):<30} {actual} {match}")

---

## 18. Optional Live LLM

Set `USE_LIVE_LLM = True` and provide a valid `OPENAI_API_KEY` to run the same loop with a real model. The tool registry and `execute_tool_batch` stay unchanged — only the planner swaps out.

In [ ]:
USE_LIVE_LLM = False

if USE_LIVE_LLM:
    try:
        from openai import OpenAI

        client = OpenAI(api_key=OPENAI_API_KEY)

        class LiveLLMPlanner:
            def plan(self, user_text: str, messages: list[dict[str, Any]]) -> dict[str, Any]:
                response = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=messages,
                    tools=TOOL_SCHEMAS,
                    tool_choice="auto",
                    temperature=0,
                )
                msg = response.choices[0].message
                result: dict[str, Any] = {"role": "assistant", "content": msg.content}
                if msg.tool_calls:
                    result["tool_calls"] = [
                        {
                            "id": tc.id,
                            "type": "function",
                            "function": {"name": tc.function.name, "arguments": tc.function.arguments},
                        }
                        for tc in msg.tool_calls
                    ]
                return result

        live_agent = MultiToolAgent(planner=LiveLLMPlanner())
        live_result = live_agent.run("Search refund policy and compute 10 + 5")
        print(live_result["answer"])
        print("Trace tool calls:", [e.get("tool_calls") for e in live_result["trace"] if e["node"] == "model"])
    except ImportError:
        print("Install openai: pip install openai")
else:
    print("Offline mode — set USE_LIVE_LLM = True to use OpenAI.")
    print("Tool schemas ready:", [s["function"]["name"] for s in TOOL_SCHEMAS])

---

## 19. Quiz

1. What are the two nodes in a basic ReAct loop?
2. Why must you loop over all `tool_calls` in one assistant message?
3. When is parallel execution safe vs when do you need a second turn?
4. What three fields besides `messages` should production agent state track?
5. What should happen when `MAX_ITERATIONS` is reached?

---

## 20. Summary

| Concept | Takeaway |
|---------|----------|
| Agent loop | Model ↔ tools until done or capped |
| Parallel calls | Execute all `tool_calls` per turn before next model step |
| Dependencies | Model chains turns using prior observations |
| ACI envelope | Consistent `[STATUS]` observations aid multi-tool transcripts |
| Trace | Log every model and tool step with latency |
| MAX_ITERATIONS | Prevent runaway cost; return partial answer + trace |
| Eval | Test full tool **chains**, not single-tool cases only |

The registry (`TOOL_REGISTRY`), batch executor (`execute_tool_batch`), and loop (`MultiToolAgent`) are the three pieces you reuse in production — swap the planner for a live LLM when ready.